In [9]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

D1 = Path("../data/d1/matches")
D2 = Path("../data/d2")

def info_partido(ruta):
    d = json.load(open(ruta, encoding="utf-8"))
    g = d["general"]
    # matchName viene como "Germany-vs-Paraguay_Mon, Jun 29, 2026, 20:30 UTC"
    texto_fecha = g["matchName"].split("_", 1)[1].replace(" UTC", "")
    fecha = pd.to_datetime(texto_fecha, format="%a, %b %d, %Y, %H:%M")
    return {"fotmob_id": int(g["matchId"]), "fecha": str(fecha.date()),
            "home": g["homeTeam"]["name"], "away": g["awayTeam"]["name"]}

d1 = pd.DataFrame([info_partido(f) for f in sorted(D1.glob("*.json"))])
d2 = pd.read_csv(D2 / "matches_detailed.csv")[["match_id", "date", "home_team_name", "away_team_name"]]

nombres_d1 = set(d1.home) | set(d1.away)
nombres_d2 = set(d2.home_team_name) | set(d2.away_team_name)
print("Solo en D1:", sorted(nombres_d1 - nombres_d2))
print("Solo en D2:", sorted(nombres_d2 - nombres_d1))

Solo en D1: ['Cape Verde', 'Curacao', 'DR Congo', 'Iran', 'Ivory Coast', 'Turkiye']
Solo en D2: ['Cabo Verde', 'Congo DR', 'Curaçao', "Côte d'Ivoire", 'IR Iran', 'Türkiye']


In [3]:
equiv = {"Cape Verde": "Cabo Verde", "Curacao": "Curaçao", "DR Congo": "Congo DR",
         "Iran": "IR Iran", "Ivory Coast": "Côte d'Ivoire", "Turkiye": "Türkiye"}
d1["home"] = d1["home"].replace(equiv)
d1["away"] = d1["away"].replace(equiv)

# volver a comparar: ambas listas deberían quedar vacías
nombres_d1 = set(d1.home) | set(d1.away)
nombres_d2 = set(d2.home_team_name) | set(d2.away_team_name)
print(sorted(nombres_d1 ^ nombres_d2))

[]


In [5]:
d1["par"] = [tuple(sorted(p)) for p in zip(d1.home, d1.away)]
d2["par"] = [tuple(sorted(p)) for p in zip(d2.home_team_name, d2.away_team_name)]

cand = d1.merge(d2, on="par")      # todas las combinaciones con el mismo par de equipos
cand["dif_dias"] = (pd.to_datetime(cand.fecha) - pd.to_datetime(cand.date)).dt.days
print(cand.dif_dias.value_counts())

union = cand[cand.dif_dias.abs() <= 1].copy()
print("filas:", len(union),
      "| partidos D1:", union.fotmob_id.nunique(),
      "| partidos D2:", union.match_id.nunique())

union["invertido"] = union.home != union.home_team_name
print("partidos con el local invertido:", union.invertido.sum())

dif_dias
 0    62
 1    24
-1    18
Name: count, dtype: int64
filas: 104 | partidos D1: 104 | partidos D2: 104
partidos con el local invertido: 0


In [6]:
union[["fotmob_id", "match_id", "invertido"]].to_csv("../data/processed/mapa_d1_d2.csv", index=False)

In [10]:
estado = pd.read_csv("../data/processed/estado_d1.csv")
mapa = pd.read_csv("../data/processed/mapa_d1_d2.csv")
X2 = pd.read_csv("../data/d2/match_prediction_features.csv")

# contexto previo al partido, siempre como "A comparado con B"
ctx = pd.DataFrame({
    "match_id":    X2.match_id,
    "elo_diff":    (X2.home_elo - X2.away_elo) / 100,
    "rank_diff":   np.log(X2.away_fifa_rank) - np.log(X2.home_fifa_rank),
    "value_diff":  np.log(X2.home_squad_total_value_eur) - np.log(X2.away_squad_total_value_eur),
    "host_diff":   X2.home_is_host - X2.away_is_host,
    "rest_diff":   X2.home_rest_days - X2.away_rest_days,
    "is_knockout": X2.is_knockout,
})

In [11]:
inst = estado.merge(mapa, on="fotmob_id").merge(ctx, on="match_id")
assert not inst.invertido.any()     # confirmamos que no hay que intercambiar A y B

inst["goal_diff"]    = inst.goles_A - inst.goles_B
inst["gd_x_t"]       = inst.goal_diff * inst.t / 90
inst["xg_diff"]      = inst.xg_A - inst.xg_B
inst["tiros_diff"]   = inst.tiros_A - inst.tiros_B
inst["al_arco_diff"] = inst.al_arco_A - inst.al_arco_B

print(inst.shape)
print("valores faltantes:", inst.isna().sum().sum())
inst[["t", "goal_diff", "xg_diff", "tiros_diff", "elo_diff", "value_diff", "y"]].describe().round(2)

(520, 24)
valores faltantes: 0


,t,goal_diff,xg_diff,tiros_diff,elo_diff,value_diff,y
count,520.00,520.00,520.00,520.00,520.00,520.00,520.00
mean,45.00,0.12,0.16,1.37,0.57,0.34,0.84
std,21.23,1.18,0.86,5.64,2.36,1.37,0.83
min,15.00,-4.00,-3.04,-25.00,-5.40,-2.82,0.00
25%,30.00,-1.00,-0.18,-2.00,-1.12,-0.79,0.00
50%,45.00,0.00,0.12,1.00,0.60,0.54,1.00
75%,60.00,1.00,0.63,4.00,2.26,1.22,2.00
max,75.00,5.00,3.14,19.00,5.10,3.21,2.00


In [12]:
inst.to_csv("../data/processed/instances.csv", index=False)